In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

df = pd.read_csv("../dataset/burnout_dataset.csv")

# df.head()
# df.info()
# df.describe()

df["task_load"] = df["task_load"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

X = df.drop(["burnout_score", "burnout_risk"], axis=1)
y_class = df["burnout_risk"]
y_score = df["burnout_score"]

X_train, X_test, y_class_train, y_class_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)

# Align regression target with same split
y_score_train = y_score.loc[X_train.index]
y_score_test = y_score.loc[X_test.index]

clf_model = RandomForestClassifier()

clf_model.fit(X_train, y_class_train)

y_pred = clf_model.predict(X_test)

print("Classification Accuracy:", accuracy_score(y_class_test, y_pred))
print(classification_report(y_class_test, y_pred))

reg_model = RandomForestRegressor()

reg_model.fit(X_train, y_score_train)

y_score_pred = reg_model.predict(X_test)

print("MAE:", mean_absolute_error(y_score_test, y_score_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_score_test, y_score_pred)))

importance = pd.Series(clf_model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

# Save classification model
with open("../backend/model/burnout_model.pkl", "wb") as f:
    pickle.dump(clf_model, f)

# Save regression model
with open("../backend/model/burnout_score_model.pkl", "wb") as f:
    pickle.dump(reg_model, f)



Classification Accuracy: 0.88
              precision    recall  f1-score   support

        High       0.88      0.99      0.93       149
         Low       0.00      0.00      0.00         1
      Medium       0.91      0.58      0.71        50

    accuracy                           0.88       200
   macro avg       0.59      0.52      0.54       200
weighted avg       0.88      0.88      0.87       200



c:\Users\adhik\Desktop\AI Burnout Predictor\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\adhik\Desktop\AI Burnout Predictor\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\adhik\Desktop\AI Burnout Predictor\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

MAE: 3.4588
RMSE: 4.616060224910416
work_hours         0.252610
stress_level       0.209423
sleep_hours        0.127261
interruptions      0.123382
commits_count      0.051267
screen_time        0.050281
breaks_count       0.048170
tasks_completed    0.046898
focus_hours        0.036295
meeting_hours      0.031818
task_load          0.022595
dtype: float64
